# Baselines

This is a code extended from Woody's code. 

- Add KNN baselines with the same metrics
- Add meter distance error scores


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [9]:
DATA_DIR = Path('data/UjiIndoorLoc')
TRAIN_CSV = DATA_DIR / 'TrainingData.csv'
VAL_CSV = DATA_DIR / 'ValidationData.csv'

train = pd.read_csv(TRAIN_CSV)
val = pd.read_csv(VAL_CSV)
print('train shape:', train.shape)
print('val shape:', val.shape)

train shape: (19937, 529)
val shape: (1111, 529)


In [10]:
train.iloc[:5,-10:-5]

,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID
0,100,-7541.2643,4.864921e+06,2,1
1,100,-7536.6212,4.864934e+06,2,1
2,100,-7519.1524,4.864950e+06,2,1
3,100,-7524.5704,4.864934e+06,2,1
4,100,-7632.1436,4.864982e+06,0,0


In [11]:
# Data definition check (UJIIndoorLoc):
# - WAP001..WAP520 are RSSI features.
# - 100 means AP not detected (no-signal sentinel).
# - Metadata columns include LONGITUDE, LATITUDE, FLOOR, BUILDINGID, etc.
wap_cols = [c for c in train.columns if c.startswith('WAP')]
meta_cols = [c for c in train.columns if not c.startswith('WAP')]
print('n_wap:', len(wap_cols))
print('meta cols:', meta_cols)
print('unique USERID (train):', train['USERID'].nunique())
print('unique USERID (val):', val['USERID'].nunique())
print('WAP min/max in train:', train[wap_cols].to_numpy().min(), train[wap_cols].to_numpy().max())
print('fraction no-signal (==100):', (train[wap_cols].to_numpy() == 100).mean())

n_wap: 520
meta cols: ['LONGITUDE', 'LATITUDE', 'FLOOR', 'BUILDINGID', 'SPACEID', 'RELATIVEPOSITION', 'USERID', 'PHONEID', 'TIMESTAMP']
unique USERID (train): 18
unique USERID (val): 1
WAP min/max in train: -104 100
fraction no-signal (==100): 0.965394550526466


In [25]:
def preprocess_wap(df: pd.DataFrame, wap_columns: list[str]) -> np.ndarray:
    # Replace no-signal sentinel with a realistic very-weak RSSI value.
    X = df[wap_columns].replace(100, -110).to_numpy(dtype=np.float32)
    return X

X_train = preprocess_wap(train, wap_cols)
X_val = preprocess_wap(val, wap_cols)

# Predict (BUILDINGID, FLOOR) jointly as a multiclass target.
y_train_raw = train['BUILDINGID'].astype(str) + '_' + train['FLOOR'].astype(str)
y_val_raw = val['BUILDINGID'].astype(str) + '_' + val['FLOOR'].astype(str)

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val = le.transform(y_val_raw)

print('X_train:', X_train.shape, 'X_val:', X_val.shape)
print('classes:', len(le.classes_))

X_train: (19937, 520) X_val: (1111, 520)
classes: 13


In [26]:
model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le.classes_),
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
pred = model.predict(X_val)

joint_acc = accuracy_score(y_val, pred)
pred_label = le.inverse_transform(pred)
pred_building = pd.Series(pred_label).str.split('_').str[0].astype(int).to_numpy()
pred_floor = pd.Series(pred_label).str.split('_').str[1].astype(int).to_numpy()

b_acc = accuracy_score(val['BUILDINGID'].to_numpy(), pred_building)
f_acc = accuracy_score(val['FLOOR'].to_numpy(), pred_floor)

print(f'Joint (building+floor) accuracy: {joint_acc:.4f}')
print(f'Building accuracy: {b_acc:.4f}')
print(f'Floor accuracy: {f_acc:.4f}')

Joint (building+floor) accuracy: 0.8920
Building accuracy: 0.9793
Floor accuracy: 0.8947


# KNN Baseline

This is KNN version. 
NOTE: We have to check the quality and baseline score.


In [15]:
from sklearn.preprocessing import StandardScaler

# X_train and X_val must already exist (your preprocess_wifi_data outputs them)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

In [16]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_cls = KNeighborsClassifier(n_neighbors=5, weights="distance", metric="euclidean")
knn_cls.fit(X_train_scaled, y_train)
pred_knn = knn_cls.predict(X_val_scaled)

joint_acc_knn = accuracy_score(y_val, pred_knn)
pred_label_knn = le.inverse_transform(pred_knn)
pred_building_knn = pd.Series(pred_label_knn).str.split('_').str[0].astype(int).to_numpy()
pred_floor_knn = pd.Series(pred_label_knn).str.split('_').str[1].astype(int).to_numpy()

b_acc_knn = accuracy_score(val['BUILDINGID'].to_numpy(), pred_building_knn)
f_acc_knn = accuracy_score(val['FLOOR'].to_numpy(), pred_floor_knn)

print(f'KNN Joint (building+floor) accuracy: {joint_acc_knn:.4f}')
print(f'KNN Building accuracy: {b_acc_knn:.4f}')
print(f'KNN Floor accuracy: {f_acc_knn:.4f}')

KNN Joint (building+floor) accuracy: 0.7930
KNN Building accuracy: 0.9901
KNN Floor accuracy: 0.7966


# Meter Accuracy - Distance Error

This is the meter based accuracy score based on the distance error. 

(I need to clarify the regression models and math equations on how we calculate. )

In [24]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import numpy as np

# ---- Targets: use dataset coordinates as XY (meter-like scale)
y_train_xy = train[['LONGITUDE', 'LATITUDE']].to_numpy(dtype=np.float32)
y_val_xy   = val[['LONGITUDE', 'LATITUDE']].to_numpy(dtype=np.float32)

# Optional: center coordinates for numerical stability (does NOT change distance errors)
origin = y_train_xy.mean(axis=0, keepdims=True)
y_train_xy_c = y_train_xy - origin
y_val_xy_c   = y_val_xy - origin

def dist_stats(y_true, y_pred):
    d = np.linalg.norm(y_true - y_pred, axis=1)
    return {
        "mean_m": float(np.mean(d)),
        "median_m": float(np.median(d)),
        "rmse_m": float(np.sqrt(np.mean(d**2))),
        "p90_m": float(np.percentile(d, 90)),
        "coord_rmse": float(np.sqrt(np.mean((y_true - y_pred)**2))),  # Kaggle-style coord RMSE
    }

# ---- KNN regression baseline (uses scaled features)
knn_reg = KNeighborsRegressor(n_neighbors=5, weights="distance", metric="euclidean")
knn_reg.fit(X_train_scaled, y_train_xy_c)
pred_knn_xy = knn_reg.predict(X_val_scaled).astype(np.float32)

knn_stats = dist_stats(y_val_xy_c, pred_knn_xy)
print("\n[KNN regression] distance error (meters):")
print(knn_stats)

# ---- XGBoost regression baseline (multioutput)
xgb_base = XGBRegressor(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)
xgb_reg = MultiOutputRegressor(xgb_base)
xgb_reg.fit(X_train_scaled, y_train_xy_c)   # using scaled features is fine here too
pred_xgb_xy = xgb_reg.predict(X_val_scaled).astype(np.float32)

xgb_stats = dist_stats(y_val_xy_c, pred_xgb_xy)
print("\n[XGBoost regression] distance error (meters):")
print(xgb_stats)


[KNN regression] distance error (meters):
{'mean_m': 12.548238754272461, 'median_m': 7.3377366065979, 'rmse_m': 23.547895431518555, 'p90_m': 26.982656478881836, 'coord_rmse': 16.65087890625}

[XGBoost regression] distance error (meters):
{'mean_m': 16.217206954956055, 'median_m': 10.555505752563477, 'rmse_m': 24.33645248413086, 'p90_m': 35.00471115112305, 'coord_rmse': 17.208471298217773}
